In [4]:
%%writefile toy_tp_transformer_block.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist

def setup_distributed():
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    return rank, world_size, local_rank, device

def cleanup_distributed():
    dist.destroy_process_group()

class ToyColumnParallelLinear(nn.Module):
    def __init__(self, in_features, out_features, world_size, bias=True, gather_output=False):
        super().__init__()
        assert out_features % world_size == 0
        self.world_size = world_size
        self.gather_output = gather_output
        self.out_per_rank = out_features // world_size

        self.weight = nn.Parameter(torch.empty(in_features, self.out_per_rank))
        self.bias = nn.Parameter(torch.empty(self.out_per_rank)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.weight, mean=0.0, std=0.02)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x):
        y_local = x @ self.weight
        if self.bias is not None:
            y_local = y_local + self.bias

        if not self.gather_output:
            return y_local

        gather_list = [torch.empty_like(y_local) for _ in range(self.world_size)]
        dist.all_gather(gather_list, y_local)
        return torch.cat(gather_list, dim=-1)

class ToyRowParallelLinear(nn.Module):
    def __init__(self, in_features, out_features, world_size, bias=True):
        super().__init__()
        assert in_features % world_size == 0
        self.in_per_rank = in_features // world_size

        self.weight = nn.Parameter(torch.empty(self.in_per_rank, out_features))
        self.bias = nn.Parameter(torch.empty(out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.weight, mean=0.0, std=0.02)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x_local):
        y_partial = x_local @ self.weight
        dist.all_reduce(y_partial, op=dist.ReduceOp.SUM)
        if self.bias is not None:
            y_partial = y_partial + self.bias
        return y_partial

class TinyTPMLP(nn.Module):
    def __init__(self, hidden_size, mlp_hidden_size, world_size):
        super().__init__()
        self.fc1 = ToyColumnParallelLinear(hidden_size, mlp_hidden_size, world_size, gather_output=False)
        self.act = nn.GELU()
        self.fc2 = ToyRowParallelLinear(mlp_hidden_size, hidden_size, world_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

class TinyTPTransformerBlock(nn.Module):
    def __init__(self, hidden_size, mlp_hidden_size, world_size):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size)
        self.mlp = TinyTPMLP(hidden_size, mlp_hidden_size, world_size)

    def forward(self, x):
        residual = x
        x = self.ln(x)
        x = self.mlp(x)
        x = x + residual
        return x

class ReferenceMLP(nn.Module):
    def __init__(self, hidden_size, mlp_hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, mlp_hidden_size)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(mlp_hidden_size, hidden_size)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class ReferenceBlock(nn.Module):
    def __init__(self, hidden_size, mlp_hidden_size):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_size)
        self.mlp = ReferenceMLP(hidden_size, mlp_hidden_size)

    def forward(self, x):
        residual = x
        x = self.ln(x)
        x = self.mlp(x)
        x = x + residual
        return x

@torch.no_grad()
def load_from_reference(tp_block, ref_block, rank, world_size):
    tp_block.ln.weight.copy_(ref_block.ln.weight)
    tp_block.ln.bias.copy_(ref_block.ln.bias)

    fc1_out = ref_block.mlp.fc1.out_features
    fc1_out_per_rank = fc1_out // world_size
    c_start = rank * fc1_out_per_rank
    c_end = (rank + 1) * fc1_out_per_rank

    tp_block.mlp.fc1.weight.copy_(ref_block.mlp.fc1.weight[c_start:c_end, :].t().contiguous())
    tp_block.mlp.fc1.bias.copy_(ref_block.mlp.fc1.bias[c_start:c_end])

    fc2_in = ref_block.mlp.fc2.in_features
    fc2_in_per_rank = fc2_in // world_size
    r_start = rank * fc2_in_per_rank
    r_end = (rank + 1) * fc2_in_per_rank

    tp_block.mlp.fc2.weight.copy_(ref_block.mlp.fc2.weight[:, r_start:r_end].t().contiguous())
    tp_block.mlp.fc2.bias.copy_(ref_block.mlp.fc2.bias)

def main():
    rank, world_size, local_rank, device = setup_distributed()

    batch_size = 8
    hidden_size = 16
    mlp_hidden_size = 32

    torch.manual_seed(2026)
    x = torch.randn(batch_size, hidden_size, device=device)

    ref_block = ReferenceBlock(hidden_size, mlp_hidden_size).to(device)

    if rank == 0:
        torch.manual_seed(1234)
        with torch.no_grad():
            nn.init.normal_(ref_block.mlp.fc1.weight, mean=0.0, std=0.02)
            nn.init.zeros_(ref_block.mlp.fc1.bias)
    
            nn.init.normal_(ref_block.mlp.fc2.weight, mean=0.0, std=0.02)
            nn.init.zeros_(ref_block.mlp.fc2.bias)
    
            nn.init.ones_(ref_block.ln.weight)
            nn.init.zeros_(ref_block.ln.bias)

    for p in ref_block.parameters():
        dist.broadcast(p.data, src=0)

    tp_block = TinyTPTransformerBlock(hidden_size, mlp_hidden_size, world_size).to(device)
    load_from_reference(tp_block, ref_block, rank, world_size)

    y_ref = ref_block(x)
    y_tp = tp_block(x)

    max_err = (y_ref - y_tp).abs().max().item()
    mean_err = (y_ref - y_tp).abs().mean().item()

    if rank == 0:
        print("========== Tiny TP Transformer Block ==========")
        print(f"world_size      : {world_size}")
        print(f"input shape     : {tuple(x.shape)}")
        print(f"output shape    : {tuple(y_tp.shape)}")
        print(f"max error       : {max_err:.8f}")
        print(f"mean error      : {mean_err:.8f}")
        print("Communication map:")
        print("  LayerNorm     : local / replicated")
        print("  MLP fc1       : Column Parallel")
        print("  GELU          : local")
        print("  MLP fc2       : Row Parallel + all-reduce")
        print("  Residual add  : local")

    cleanup_distributed()

if __name__ == "__main__":
    main()

Overwriting toy_tp_transformer_block.py


In [5]:
!torchrun --standalone --nproc_per_node=2 toy_tp_transformer_block.py

W0409 10:48:15.704000 124 torch/distributed/run.py:852] 
W0409 10:48:15.704000 124 torch/distributed/run.py:852] *****************************************
W0409 10:48:15.704000 124 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0409 10:48:15.704000 124 torch/distributed/run.py:852] *****************************************
[W409 10:48:16.374127270 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W409 10:48:17.145151214 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W409 10:48:17.161398795 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
========== Tiny TP Transformer Block ==========
world_size      : 2
input shape     : (8, 16)
output shape    : (8, 16)
max error       : 0.00000012
me